# Notebook 4 — Agent-Based Model: Roastery Customer Flow Simulation

**The core of the project.** We simulate thousands of virtual customers (agents) walking through the 5-floor Chicago Roastery, making floor-to-floor transition decisions based on the calibrated transition matrix from Notebook 3.

**Each agent:**
1. Enters on a floor (sampled from entry distribution)
2. Spends time on that floor (sampled from dwell time distribution)
3. Decides: move to another floor or exit (sampled from transition matrix)
4. Repeats until exit

**Outputs:**
- Floor visit frequency and dwell time distributions
- Average floors visited per customer
- Peak congestion by floor and time
- Revenue proxy by floor (visits × average spend)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from IPython.display import display

np.random.seed(42)

ON_KAGGLE = Path('/kaggle/working').exists()
if ON_KAGGLE:
    DATA_DIR = Path('/kaggle/input/starbucks-roastery-abm')
    if not DATA_DIR.exists():
        DATA_DIR = Path('/kaggle/input/datasets/shiratoriseto/starbucks-roastery-abm')
    PROC_DIR = Path('/kaggle/working')
else:
    DATA_DIR = Path('../data/raw')
    PROC_DIR = Path('../data/processed')

floor_labels = ['1F', '2F', '3F', '4F', '5F']

print('Loading calibrated data...')

# Load floor data
floors_df = pd.read_csv(DATA_DIR / 'roastery_floors.csv')
menu_df = pd.read_csv(DATA_DIR / 'roastery_menu.csv')

print(f'Floors: {len(floors_df)} | Menu items: {len(menu_df)}')
print('Setup complete.')

## Section 1 — ABM Configuration

All parameters in one place for transparency and reproducibility.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# ABM CONFIGURATION
# ══════════════════════════════════════════════════════════════════════

N_AGENTS = 5000  # Number of simulated visitors
MAX_FLOORS_PER_VISIT = 8  # Safety cap to prevent infinite loops

# Transition matrix (calibrated from Notebook 3)
# Using hardcoded values from calibration to be self-contained
# Prior: physical adjacency, Observed: review NLP, Blended with alpha
TRANSITION_MATRIX = np.array([
    # To: 1F     2F     3F     4F     5F     EXIT
    [0.00, 0.35, 0.12, 0.04, 0.02, 0.47],  # From 1F
    [0.18, 0.00, 0.30, 0.10, 0.04, 0.38],  # From 2F
    [0.10, 0.15, 0.00, 0.30, 0.07, 0.38],  # From 3F
    [0.06, 0.06, 0.15, 0.00, 0.22, 0.51],  # From 4F
    [0.10, 0.05, 0.10, 0.25, 0.00, 0.50],  # From 5F
])

# Entry distribution (from review sequences)
ENTRY_PROBS = np.array([0.50, 0.20, 0.10, 0.15, 0.05])  # Most enter on 1F

# Dwell time per floor (minutes, mean and std)
DWELL_TIME = {
    '1F': {'mean': 15, 'std': 8},   # Quick browse/retail
    '2F': {'mean': 25, 'std': 12},  # Eating takes time
    '3F': {'mean': 20, 'std': 10},  # Coffee experience
    '4F': {'mean': 35, 'std': 15},  # Bar, lingering
    '5F': {'mean': 15, 'std': 8},   # Rooftop, seasonal
}

# Average spend per floor visit ($)
AVG_SPEND = {
    '1F': 12.0,   # Coffee + pastry or merch
    '2F': 15.0,   # Food-focused
    '3F': 14.0,   # Specialty coffee/flights
    '4F': 22.0,   # Cocktails are premium
    '5F': 10.0,   # Lighter offerings
}

print(f'Agents: {N_AGENTS:,}')
print(f'Entry distribution: {dict(zip(floor_labels, ENTRY_PROBS))}')
print(f'Transition matrix shape: {TRANSITION_MATRIX.shape}')
print(f'\nDwell times (mean min):')
for f in floor_labels:
    print(f'  {f}: {DWELL_TIME[f]["mean"]} ± {DWELL_TIME[f]["std"]} min')

## Section 2 — Agent Simulation Engine

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# ABM SIMULATION ENGINE
# ══════════════════════════════════════════════════════════════════════

def simulate_agent(agent_id):
    """Simulate one customer's journey through the Roastery."""
    # Entry floor
    current_floor_idx = np.random.choice(5, p=ENTRY_PROBS)
    
    visits = []
    total_time = 0
    total_spend = 0
    
    for step in range(MAX_FLOORS_PER_VISIT):
        floor = floor_labels[current_floor_idx]
        
        # Dwell time (truncated normal, min 3 minutes)
        dwell = max(3, np.random.normal(DWELL_TIME[floor]['mean'], DWELL_TIME[floor]['std']))
        
        # Spend
        spend = max(0, np.random.normal(AVG_SPEND[floor], AVG_SPEND[floor] * 0.3))
        
        visits.append({
            'agent_id': agent_id,
            'step': step,
            'floor': floor,
            'dwell_min': round(dwell, 1),
            'spend': round(spend, 2),
            'cumulative_time': round(total_time + dwell, 1),
        })
        
        total_time += dwell
        total_spend += spend
        
        # Transition decision
        transition_probs = TRANSITION_MATRIX[current_floor_idx]  # includes exit
        options = list(range(5)) + ['exit']
        choice = np.random.choice(6, p=transition_probs)
        
        if choice == 5:  # Exit
            break
        else:
            current_floor_idx = choice
    
    return visits, total_time, total_spend

# Run simulation
all_visits = []
agent_summaries = []

for i in range(N_AGENTS):
    visits, total_time, total_spend = simulate_agent(i)
    all_visits.extend(visits)
    agent_summaries.append({
        'agent_id': i,
        'n_floors_visited': len(visits),
        'total_time_min': round(total_time, 1),
        'total_spend': round(total_spend, 2),
        'floors_visited': '|'.join([v['floor'] for v in visits]),
        'entry_floor': visits[0]['floor'],
        'exit_floor': visits[-1]['floor'],
    })
    
    if (i + 1) % 1000 == 0:
        print(f'  Simulated {i+1:,} / {N_AGENTS:,} agents')

visits_df = pd.DataFrame(all_visits)
agents_df = pd.DataFrame(agent_summaries)

print(f'\nSimulation complete!')
print(f'Total visits: {len(visits_df):,}')
print(f'Avg floors per agent: {agents_df.n_floors_visited.mean():.2f}')
print(f'Avg time per agent: {agents_df.total_time_min.mean():.1f} min')
print(f'Avg spend per agent: ${agents_df.total_spend.mean():.2f}')

## Section 3 — Floor-Level Analysis

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# FLOOR-LEVEL ANALYSIS
# ══════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['#00704A', '#1E3932', '#D4E9E2', '#CBA258', '#87CEEB']

# 1. Visit frequency by floor
floor_visit_counts = visits_df['floor'].value_counts().reindex(floor_labels)
axes[0, 0].bar(floor_labels, floor_visit_counts.values, color=colors, edgecolor='white')
axes[0, 0].set_title('Total Visits by Floor', fontweight='bold')
axes[0, 0].set_ylabel('Visit Count')
for i, v in enumerate(floor_visit_counts.values):
    axes[0, 0].text(i, v + 50, f'{v:,}\n({v/N_AGENTS:.1f}/agent)', ha='center', fontsize=9)

# 2. Dwell time distribution by floor
for i, floor in enumerate(floor_labels):
    mask = visits_df['floor'] == floor
    axes[0, 1].hist(visits_df[mask]['dwell_min'], bins=20, alpha=0.6, label=floor, color=colors[i])
axes[0, 1].set_title('Dwell Time Distribution by Floor', fontweight='bold')
axes[0, 1].set_xlabel('Minutes')
axes[0, 1].legend()

# 3. Revenue by floor
floor_revenue = visits_df.groupby('floor')['spend'].sum().reindex(floor_labels)
axes[1, 0].bar(floor_labels, floor_revenue.values, color=colors, edgecolor='white')
axes[1, 0].set_title('Total Revenue by Floor ($)', fontweight='bold')
axes[1, 0].set_ylabel('Revenue ($)')
for i, v in enumerate(floor_revenue.values):
    axes[1, 0].text(i, v + 500, f'${v:,.0f}', ha='center', fontsize=9)

# 4. Floors visited distribution
floor_dist = agents_df['n_floors_visited'].value_counts().sort_index()
axes[1, 1].bar(floor_dist.index, floor_dist.values, color='#00704A', edgecolor='white')
axes[1, 1].set_title('Number of Floors Visited per Agent', fontweight='bold')
axes[1, 1].set_xlabel('Floors Visited')
axes[1, 1].set_ylabel('Number of Agents')

plt.suptitle(f'ABM Simulation Results (n={N_AGENTS:,} agents)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary table
floor_summary = visits_df.groupby('floor').agg(
    total_visits=('agent_id', 'size'),
    avg_dwell=('dwell_min', 'mean'),
    total_revenue=('spend', 'sum'),
    avg_spend=('spend', 'mean'),
).reindex(floor_labels).round(2)
floor_summary['visits_per_agent'] = (floor_summary['total_visits'] / N_AGENTS).round(2)
floor_summary['revenue_share'] = (floor_summary['total_revenue'] / floor_summary['total_revenue'].sum() * 100).round(1)

print('\n=== Floor Summary ===')
display(floor_summary)

## Section 4 — Customer Journey Patterns

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CUSTOMER JOURNEY PATTERNS
# ══════════════════════════════════════════════════════════════════════

# Most common journey paths
path_counts = agents_df['floors_visited'].value_counts().head(15)

print('=== Top 15 Customer Journeys ===')
for path, count in path_counts.items():
    pct = count / N_AGENTS * 100
    path_display = path.replace('|', ' → ')
    print(f'  {path_display:40s}  {count:>5,} agents ({pct:.1f}%)')

# Entry → Exit flow
print('\n=== Entry Floor → Exit Floor ===')
flow = agents_df.groupby(['entry_floor', 'exit_floor']).size().unstack(fill_value=0)
display(flow)

# Journey length by entry floor
print('\n=== Avg Journey Length by Entry Floor ===')
entry_summary = agents_df.groupby('entry_floor').agg(
    n_agents=('agent_id', 'size'),
    avg_floors=('n_floors_visited', 'mean'),
    avg_time=('total_time_min', 'mean'),
    avg_spend=('total_spend', 'mean'),
).round(2)
display(entry_summary)

## Section 5 — Sankey Flow Diagram

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SANKEY FLOW DIAGRAM
# ══════════════════════════════════════════════════════════════════════

# Build flow data from all transitions
flow_counts = {}
for _, row in agents_df.iterrows():
    floors = row['floors_visited'].split('|')
    for i in range(len(floors) - 1):
        key = (floors[i], floors[i+1])
        flow_counts[key] = flow_counts.get(key, 0) + 1
    # Exit from last floor
    key = (floors[-1], 'Exit')
    flow_counts[key] = flow_counts.get(key, 0) + 1

# Entry flows
entry_flow = agents_df['entry_floor'].value_counts()
for floor, count in entry_flow.items():
    flow_counts[('Entry', floor)] = count

# Build Sankey
labels = ['Entry'] + floor_labels + ['Exit']
label_map = {l: i for i, l in enumerate(labels)}

sources, targets, values = [], [], []
for (src, tgt), count in flow_counts.items():
    if count >= 50:  # Only show significant flows
        sources.append(label_map[src])
        targets.append(label_map[tgt])
        values.append(count)

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15, thickness=20,
        label=labels,
        color=['#333'] + colors + ['#999'],
    ),
    link=dict(
        source=sources, target=targets, value=values,
        color=['rgba(0,112,74,0.3)'] * len(sources),
    ),
)])

fig.update_layout(
    title=f'Customer Flow Through Roastery (n={N_AGENTS:,} agents)',
    height=500, template='plotly_white',
)
fig.show()

## Section 6 — Save Simulation Results

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SAVE
# ══════════════════════════════════════════════════════════════════════

OUT_DIR = PROC_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

visits_df.to_csv(OUT_DIR / 'abm_visits.csv', index=False)
agents_df.to_csv(OUT_DIR / 'abm_agents.csv', index=False)
floor_summary.to_csv(OUT_DIR / 'abm_floor_summary.csv')

print(f'Saved: abm_visits.csv ({len(visits_df):,} rows)')
print(f'Saved: abm_agents.csv ({len(agents_df):,} rows)')
print(f'Saved: abm_floor_summary.csv')

print(f'\n=== Simulation Summary ===')
print(f'Agents:              {N_AGENTS:,}')
print(f'Total visits:        {len(visits_df):,}')
print(f'Avg floors/agent:    {agents_df.n_floors_visited.mean():.2f}')
print(f'Avg time/agent:      {agents_df.total_time_min.mean():.1f} min')
print(f'Avg spend/agent:     ${agents_df.total_spend.mean():.2f}')
print(f'Total simulated rev: ${agents_df.total_spend.sum():,.0f}')

## Model Assumptions & Limitations

| Assumption | Reality | Impact |
|---|---|---|
| Agents are independent | Real visitors travel in groups | Underestimates group dynamics |
| Uniform arrival time | Real arrivals peak at 10AM-2PM | No congestion modeling |
| Transition matrix is static | May vary by time of day, season | Missing temporal dynamics |
| Dwell time is normally distributed | May be bimodal (quick browse vs. linger) | Smooths over real patterns |
| Average spend is per-floor | Real spend depends on what's ordered | Approximation |
| Exit probability is constant | Fatigue increases exit probability over time | May underestimate long visits |

**What this model IS:** A transparent, reproducible simulation framework grounded in real review data.

**What this model IS NOT:** A prediction of actual customer behavior. With real WiFi/camera/POS data, every parameter could be calibrated precisely.